# Corrected DPE: strict nested selection and exact threshold controls

**Self-contained code: upload this notebook to Kaggle; no code dataset is required.**
Turn **Internet On**, select a **CPU runtime**, then **Save & Run All**.
The experiment uses CPU head fitting; GPU acceleration is not required.

Uses the same seven saved feature sets (five standard ERM, two ERM*) as
`noerula/dpe-nested-code`, recovered from attached inputs or the existing public
output API. No backbones are retrained. Missing feature sets cause an explicit stop.

## Declared corrected protocol

- Five inner evaluation folds. For each fold, rebuild five OOF ensembles and
  a full-refit ensemble **using that inner training pool only**. The entire
  inner evaluation fold is excluded from every head and calibration fit.
- Inner evaluation uses the complete prediction procedure used on test:
  calibrated fold averaging, post-average bias/thresholding, or OOF calibration
  transferred to the full-refit ensemble. Each family selects independently.
- Preserve the original four (loss, penalty) settings: average/0.1,
  maximum/0.1, average/0.01, maximum/0.01. Fixed setting: average/0.1.
  **Primary selection: balanced NLL; secondary: WGA.** Original tie breakers remain.
- Direct WGA thresholds search every attainable decision partition, with
  equal scores moved together. They are applied **after averaging probabilities**.
  Foldwise bias calibration is reported under its own explicit name.
- Keep full-validation DPE, a 75-head bag, cross-fit uniform, pooled 60/40
  affine/linear controls, and DFR-lite. DFR standardization is fitted inside CV.
- All ten split seeds and seven sources are declared. Folds are not independent
  backbones; intervals use per-source split means. ERM* has only two sources.
- The test set and source backbones were previously inspected. This remains
  **exploratory**, even with the corrected selection pipeline.

Expect substantially more CPU time than the original run: **41 head ensembles
per source/split instead of 11** (each ensemble contains 15 heads). Stage caches
permit resuming after a time limit; no completed old-protocol results are reused.

In [ ]:
# Keep these declared seeds/rules fixed while running this experiment.
SPLITS = list(range(10))
EXPERIMENT_TIME_LIMIT_HOURS = 8.5
WORKERS = 4  # capped at the number of available CPU cores below
RUN_SAVED_REGRESSION_CHECK = True

import os, sys, json, shutil, subprocess, zipfile, hashlib, platform
from pathlib import Path
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
if not WORK.is_dir():
    raise SystemExit('Run this notebook in Kaggle. The code ZIP also supports local CLI use.')
STATE = WORK / 'corrected_dpe_state'
OUT = STATE / 'nested_v2'
CODE = WORK / 'corrected_code_v2'
RECOVERED = STATE / 'recovered_sources'
STATE.mkdir(parents=True, exist_ok=True)
WORKERS = max(1, min(WORKERS, os.cpu_count() or 1))

def sh(args, check=True):
    args = [str(a) for a in args]
    print('$', ' '.join(args), flush=True)
    process = subprocess.Popen(args, cwd=CODE, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if check and rc:
        raise RuntimeError(f'Command failed ({rc}): {args}')
    return rc


In [ ]:
import base64, io
BUNDLE_SHA256 = 'd5f0f59e2d4546c3fe6328f6369f6c5ab7d659404b91af0a9de0eb38a855f3bd'
# Embedded source ZIP; a separate copy is provided for review.
BUNDLE_B64 = (
'...'
'iQUAAAA='
)
bundle_bytes = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(bundle_bytes).hexdigest() == BUNDLE_SHA256, 'Embedded bundle changed'
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as bundle:
    for member in bundle.infolist():
        target = (CODE / member.filename).resolve()
        if not target.is_relative_to(CODE.resolve()):
            raise ValueError('Unsafe archive path')
    bundle.extractall(CODE)
print('Verified embedded source bundle:', BUNDLE_SHA256)


In [ ]:
# Preflight: software versions and the regression tests (no Waterbirds fits).
import importlib
required = ('numpy', 'scipy', 'sklearn', 'torch', 'torchvision', 'yaml', 'pandas', 'tqdm', 'PIL')
for name in required:
    mod = importlib.import_module(name)
    print(f'{name:12} {getattr(mod, "__version__", "?")}')
print('Python:', platform.python_version(), '| CPU workers:', WORKERS)
sh([sys.executable, 'corrected_selftest.py'])
sys.path.insert(0, str(CODE))
import corrected_nested as corrected
print('Protocol:', corrected.PROTOCOL)


## Resume an interrupted corrected run

Attach its Kaggle output, or upload `corrected_nested_resume.zip` as a Dataset
(Kaggle can extract it). This cell chooses one compatible state with the most
progress. Old `e2e_state/nested` results are never imported. If software/source
fingerprints differ, the run stops instead of silently mixing results.

In [ ]:
# Support an uploaded resume ZIP when Kaggle has not already extracted it.
resume_zips = sorted(INPUT.rglob('corrected_nested_resume.zip'))
resume_roots = []
for number, path in enumerate(resume_zips):
    target_root = WORK / f'resume_archive_{number}'
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            if not (target_root / member.filename).resolve().is_relative_to(target_root.resolve()):
                raise ValueError('Unsafe resume archive path')
        archive.extractall(target_root)
    resume_roots.append(target_root)
candidates = []
for root in [INPUT, *resume_roots]:
    for prior in root.rglob('corrected_dpe_state'):
        nested = prior / 'nested_v2'
        marker = nested / 'protocol.json'
        if marker.is_file():
            protocol = json.loads(marker.read_text())
            if protocol.get('protocol') == corrected.PROTOCOL and protocol.get('code_sha256') == corrected.code_identity():
                candidates.append(prior)
if candidates and not (OUT / 'protocol.json').exists():
    prior = max(candidates, key=lambda p: (len(list((p/'nested_v2/results').glob('*/split*.json'))),
                                           len(list((p/'nested_v2/checkpoints').rglob('complete.json')))))
    shutil.copytree(prior / 'nested_v2', OUT, dirs_exist_ok=True)
    print('Restored corrected checkpoints from:', prior)
else:
    print('Using current state, or starting a fresh corrected run.')
corrected.prepare_protocol(OUT)  # validates code, settings and runtime identity


In [ ]:
# Recover the SAME saved feature sets; no backbone retraining fallback.
SOURCE_INDEX = STATE / 'source_index'
roots = [INPUT, RECOVERED]
sh([sys.executable, 'nested_selection.py', 'discover', '--roots', *roots,
    '--out', SOURCE_INDEX], check=False)
source_file = SOURCE_INDEX / 'sources.json'
specs = json.loads(source_file.read_text())['sources'] if source_file.exists() else []
expected = {*(f'normal-erm-seed{i}' for i in range(5)), *(f'ermstar-seed{i}' for i in range(2))}
if {s['id'] for s in specs} != expected:
    print('Recovering missing saved features from public Kaggle outputs (Internet must be On).')
    recovery_code = 'from src.methods.dpe_independent.recovery import recover; recover(' + repr(str(RECOVERED)) + ')'
    sh([sys.executable, '-c', recovery_code])
    sh([sys.executable, 'nested_selection.py', 'discover', '--roots', *roots, '--out', SOURCE_INDEX])
    specs = json.loads(source_file.read_text())['sources']
actual = {s['id'] for s in specs}
if actual != expected or len(specs) != 7:
    raise RuntimeError(f'Expected the original seven sources. Missing={expected-actual}; unexpected={actual-expected}')
(OUT / 'sources.json').write_text(json.dumps(dict(sources=specs, blockers=[]), indent=2))
print('Sources:', sorted(actual))
if RUN_SAVED_REGRESSION_CHECK:
    sh([sys.executable, 'nested_selection.py', 'check', '--out', OUT])


## Run all declared cells

The budget is checked between head-ensemble fits. Completed stages are saved
immediately; a timeout returns a partial run that can be resumed. Do not change
settings after reading interim test results. Run all ten seeds before interpreting
the final comparison.

In [ ]:
run_rc = sh([sys.executable, 'corrected_nested.py', 'run', '--out', OUT,
             '--splits', *SPLITS, '--workers', WORKERS,
             '--time-limit-hours', EXPERIMENT_TIME_LIMIT_HOURS], check=False)
print('Run return code:', run_rc, '(0=complete, 3=budget reached, 1=error)')


In [ ]:
# Always package progress, including partial runs, before surfacing a run error.
sh([sys.executable, 'corrected_nested.py', 'summarize', '--out', OUT])
result_zip = WORK / 'corrected_nested_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and 'checkpoints' not in path.relative_to(OUT).parts:
            archive.write(path, str(path.relative_to(OUT)))
resume_zip = WORK / 'corrected_nested_resume.zip'
with zipfile.ZipFile(resume_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and not any(part.startswith('.pending-') for part in path.parts):
            archive.write(path, str(path.relative_to(WORK)))
print('Analysis bundle:', result_zip)
print('Resume bundle (includes prediction checkpoints):', resume_zip)
print('Reports:', OUT/'REPORT.md', OUT/'CORRECTED_DIAGNOSTICS.md')
completion = json.loads((OUT/'completion.json').read_text()) if (OUT/'completion.json').exists() else {}
print('Completion:', completion)
if run_rc == 3:
    print('PARTIAL: attach this output or the resume ZIP to the same notebook and Save & Run All again.')
elif run_rc != 0:
    errors = sorted((OUT/'errors').glob('*.txt'))
    for path in errors[:3]:
        print(path, '\n', path.read_text())
    raise RuntimeError('Experiment failed; progress was packaged. Inspect the error files before resuming.')
